<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_7_Camel_Cards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''32T3K 765
T55J5 684
KK677 28
KTJJT 220
QQQJA 483'''

with open('7.txt') as f: puzzle = f.read()

puzzle = puzzle.split('\n')
puzzle = [ line.split(' ') for line in puzzle ]
puzzle = { line[0] : int(line[1]) for line in puzzle }
puzzle

################################################################################
# PART 1

# assign order to hands
hand_types = ['High-card', 'One-pair', 'Two-pair', 'Three-of-a-kind', 'Full-house', 'Four-of-a-kind', 'Five-of-a-kind']
cards_strength = ['2', '3', '4', '5', '6', '7', '8', '9', 'T', 'J', 'Q', 'K', 'A']

def determine_hand_type(h):
    l = len(set(h))

    # AAAAA
    if l == 1:
        h_type = 'Five-of-a-kind'

    # AAAAB or AAABB
    elif l == 2:
        counts = [ h.count(c) for c in set(h) ]
        if 4 in counts:
            h_type = 'Four-of-a-kind'
        else:
            assert set(counts) == {2, 3}
            h_type = 'Full-house'

    # AAABC, AABBC
    elif l == 3:
        counts = [ h.count(c) for c in set(h) ]
        if 3 in counts:
            assert counts.count(1) == 2
            h_type = 'Three-of-a-kind'
        else:
            assert counts.count(2) == 2
            assert 1 in counts
            h_type = 'Two-pair'

    # AABCD
    elif l == 4:
        h_type = 'One-pair'

    # ABCDE
    else:
        h_type = 'High-card'

    return h_type

hands = { hand : { 'bid' : puzzle[hand], 'type' : None } for hand in puzzle }
hands

for h in hands:
    h_type = determine_hand_type(h)
    assert h_type is not None
    hands[h]['type'] = h_type


# ranking hands
def calculate_hands_ranked(key):
    global hands, cards_strength

    hands_ranked = []

    for h1 in hands:

        h1_t = hands[h1][key]
        h1_r = hand_types.index(h1_t)

        # stepping until reaching a same-or-greater hand type card
        i = 0
        while i < len(hands_ranked):

            h2 = hands_ranked[i]
            h2_t = hands[h2][key]
            h2_r = hand_types.index(h2_t)

            # hand is better, move on
            if h1_r > h2_r:
                i += 1

            # same hands, check each card
            elif h1_r == h2_r:
                move_to_next = False
                j = 0
                while j < 5:
                    if h1[j] == h2[j]:
                        j += 1
                    else:
                        h1_s = cards_strength.index(h1[j])
                        h2_s = cards_strength.index(h2[j])
                        move_to_next = (h1_s > h2_s)
                        break
                assert j != 5
                if move_to_next:
                    i += 1
                else:
                    break

            # hand is weaker, stop
            else:
                break

        # insert cards in rank order
        if i == len(hands_ranked):
            hands_ranked.append(h1)
        else:
            hands_ranked.insert(i, h1)

    return hands_ranked

# total winnings
hands_ranked = calculate_hands_ranked('type')
for h in hands: hands[h]['rank'] = hands_ranked.index(h) + 1
sum( hands[h]['rank'] * hands[h]['bid'] for h in hands )

################################################################################
# PART 2

# redefining strength of 'J'
cards_strength = ['J', '2', '3', '4', '5', '6', '7', '8', '9', 'T', 'Q', 'K', 'A']

# calculating best scenarios with J
for h in hands:

    if 'J' not in h:
        hands[h]['best_type'] = hands[h]['type']
    else:
        c = h.count('J')

        # JJJJJ or AJJJJ
        if c in [4, 5]:
            hands[h]['best_type'] = 'Five-of-a-kind'

        # AAJJJ or ABJJJ
        elif c == 3:
            if hands[h]['type'] == 'Full-house': hands[h]['best_type'] = 'Five-of-a-kind'
            else:
                assert hands[h]['type'] == 'Three-of-a-kind'
                hands[h]['best_type'] = 'Four-of-a-kind'

        # AAAJJ or AABJJ or ABCJJ
        elif c == 2:
            if hands[h]['type'] == 'Full-house': hands[h]['best_type'] = 'Five-of-a-kind'
            elif hands[h]['type'] == 'Two-pair': hands[h]['best_type'] = 'Four-of-a-kind'
            else:
                assert hands[h]['type'] == 'One-pair'
                hands[h]['best_type'] = 'Three-of-a-kind'

        # AAAAJ-AAABJ-AABBJ-AABCJ-ABCDJ
        else:
            assert c == 1
            if hands[h]['type'] == 'Four-of-a-kind': hands[h]['best_type'] = 'Five-of-a-kind'
            elif hands[h]['type'] == 'Three-of-a-kind': hands[h]['best_type'] = 'Four-of-a-kind'
            elif hands[h]['type'] == 'Two-pair': hands[h]['best_type'] = 'Full-house'
            elif hands[h]['type'] == 'One-pair': hands[h]['best_type'] = 'Three-of-a-kind'
            else:
                assert hands[h]['type'] == 'High-card'
                hands[h]['best_type'] = 'One-pair'

hands

# total winnings
hands_ranked = calculate_hands_ranked('best_type')
for h in hands: hands[h]['rank'] = hands_ranked.index(h) + 1
sum( hands[h]['rank'] * hands[h]['bid'] for h in hands )